# Lesson 83 · A Transformer from scratch — the one formula holding up the age of large models: multi-head attention, rebuilt by hand

**Learning goals**
1. Say clearly who Q / K / V each are (their ID cards)
2. Implement `scaled_dot_product_attention` and understand the $\sqrt{d_k}$ scaling
3. Know what multi-head is doing (several independent sets of attention)

> **Structure**: first explain attention in plain QKV terms, then look at the formula and a worked example by hand; after that we unpack multi-head and positional encoding, and compare with the Whisper encoder (L70).

🔗 **Aurora connection**: attention is the shared foundation of the Whisper encoder and the local LLM decoder.

← **Previous lesson** · [L82 · Music features visualized](../8_music/L82_visual_music.ipynb)

> Last lesson covered **music features visualized**: chromagrams, beat grids and similarity heatmaps on display.
> This lesson explores **a Transformer from scratch**.

## This lesson's story: why does a translation machine need to see the whole sentence at once?

When a classic RNN (LSTM) translates "The cat that the dog chased up the old oak tree yesterday finally came home", it reads "The" first and stores it in the hidden state; then it reads "cat" and updates the hidden state… but by the time it reaches "came home", the information about "cat" has been diluted over and over — this is called the **long-range dependency problem**.

**The Transformer's fix**: instead of making the words queue up and be read one at a time, let every word see all the words in the sentence at once.

**The intuition behind attention**:

```
You read the word "bank":
  Context 1: "I deposited money in the bank" → bank = a financial institution (attends to "money")
  Context 2: "The fish swam near the bank"   → bank = a riverbank (attends to "fish", "swam")
```

The same word adjusts its meaning dynamically according to its context — that is "attention". Mathematically:

```
Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V
```

- **Q (Query)**: the current word is "asking": who should I pay attention to?
- **K (Key)**: every word "answers": how relevant I am to you
- **V (Value)**: the content that actually gets read
- **√d_k scaling**: stops the dot product's variance from blowing up in high dimensions → softmax saturating → vanishing gradients

**Multi-head**: run h attention heads in parallel in the same layer, so different heads can keep an eye on grammar, meaning and position separately.

This lesson's task: implement `scaled_dot_product_attention(Q, K, V, mask=None)` and check numerically that it matches the reference implementation.

## 🤔 Why did engineers invent it?

- **What happens without it?** An RNN reads a sentence word by word, in series; by the end of the sentence the first words have been diluted again and again, so long sentences keep "losing words" in translation — and it has to compute one word at a time, so it can't be sped up in parallel.
- **What real problem does it solve?** Attention lets every word "see" any word in the whole sentence directly, in a single step, using similarity to decide where to look — long-range dependencies and parallel training solved in one go.
- **Where does it come up again later?** L84's LoRA attaches right next to attention's projection matrices, L85's KV-Cache caches exactly attention's K/V, L90's agent reasons with it, and L91 draws the attention weights as a heatmap. It is the same foundation under every large-model lesson that follows.

## ⚠️ Common misconceptions

> Don't think of attention as "some mysterious weighted average". What it really does is: first use **Q·Kᵀ** to compute similarity scores between words, then use **softmax** to turn the scores into weights — the weights are decided entirely by similarity, with no magic at all.

In [ ]:
import numpy as np

### 🔢 Work it by hand on a 3×3 first (before going high-dimensional, understand the mechanism with small integers)

You can't do high-dimensional vectors in your head, so let's first walk through the whole process with **3 tokens, 2-D vectors, all small integers**. Say we want to compute the attention output for one query:

```
Q  = [1, 1]                 # the query vector at the current position

K1 = [2, 0]   V1 = [1, 0]   # token 1's key / value
K2 = [0, 1]   V2 = [0, 1]   # token 2
K3 = [0, 0]   V3 = [1, 1]   # token 3
```

**Step 1 · similarity scores = Q·Kᵀ** (dot Q with every K)

```
Q·K1 = 1×2 + 1×0 = 2
Q·K2 = 1×0 + 1×1 = 1
Q·K3 = 1×0 + 1×0 = 0
scores = [2, 1, 0]          # Q is most like K1 → highest score
```

> (The real computation also divides by √d_k here; for clarity by hand we skip the scaling in this example — the mechanism is exactly the same.)

**Step 2 · softmax turns the scores into weights** (take e^score, then normalize so they sum to 1)

```
e^2 ≈ 7.39,  e^1 ≈ 2.72,  e^0 = 1     total ≈ 11.11
weights ≈ [7.39/11.11, 2.72/11.11, 1/11.11] ≈ [0.67, 0.24, 0.09]
```

The higher the score, the bigger the weight — K1 gets 67% of the attention.

**Step 3 · weighted sum output = weights·V** (use the weights to mix the three Vs)

```
output ≈ 0.67×[1,0] + 0.24×[0,1] + 0.09×[1,1]
       ≈ [0.67+0.09, 0.24+0.09]
       ≈ [0.76, 0.33]
```

That is all there is to attention: **dot products for similarity → softmax into weights → mix V by the weights**. The code below just moves this same process into high dimensions, batches and vectorized form — the maths is identical.

## 1. Attention(Q, K, V) = softmax(QKᵀ / √d_k) · V

Let the sequence length be `seq`, and the feature dimensions be `d_k` (key/query dimension) and `d_v` (value dimension).

- **Q** (Query): what the current position wants to "look up"
- **K** (Key): the "index label" each position offers
- **V** (Value): the information each position actually carries

The computation steps:
```
scores  = Q @ Kᵀ          # (seq_q, seq_k)  dot-product similarity
scores /= sqrt(d_k)        # scaling
weights = softmax(scores)  # each row normalized into probabilities
output  = weights @ V      # (seq_q, d_v)  weighted aggregation
```

Geometrically, `Q[i] · K[j]` is the cosine of the angle between the vectors times their lengths — the closer Q and K point in the same direction, the higher that position's weight.

---

### A concrete example: resolving a word with several meanings

Processing the sentence *"The bank by the river lowered rates"*, when computing the output for the word **bank**:

| Role | Content |
|---|---|
| **Q (bank's query)** | bank queries the context: am I a financial institution or a riverbank? |
| **K[river] (river's key)** | river labels itself as a physical-geography word |
| **K[rates] (rates's key)** | rates labels itself as a finance word |
| **V[river] (river's value)** | river's semantic embedding (carrying information about water, geography and so on) |
| **V[rates] (rates's value)** | rates's semantic embedding (carrying information about interest rates, finance and so on) |

The attention weights `w[bank→river]` and `w[bank→rates]` are both fairly high, and once their Vs are added together by weight, **bank's output vector blends both contexts, the riverbank and the interest rates**, which the downstream layers use to disambiguate.

**Why separate Q/K/V**: Q and K decide "where to look", V decides "what to extract". Splitting them lets the model learn the attention rule (Q·K) and the content projection (V) separately — if you made K=V, the attention rule and the content would be welded together and expressive power would drop sharply.

In [ ]:
# Demo: 2 queries, 3 keys/values, d_k=4, d_v=6
np.random.seed(0)
Q_demo = np.random.randn(2, 4)   # (seq_q=2, d_k=4)
K_demo = np.random.randn(3, 4)   # (seq_k=3, d_k=4)
V_demo = np.random.randn(3, 6)   # (seq_k=3, d_v=6)

scores_demo = Q_demo @ K_demo.T  # (2, 3)
print('scores shape:', scores_demo.shape)
print('scores (unscaled):\n', scores_demo.round(3))

## 2. Why divide by √d_k?

Suppose the components of Q and K are independent random variables with mean 0 and variance 1. Then the dot product `Q · K = Σ Q_i K_i` has variance `d_k` and standard deviation `√d_k`.

When `d_k` is large (say 512), the dot products get large in absolute value and push softmax into its saturated region (one entry near 1, the rest near 0); the gradient is almost 0 and training stalls.

Dividing by `√d_k` normalizes the variance back to 1, keeping the size of softmax's inputs under control:
```
Var(Q · K / √d_k) = d_k / d_k = 1
```

In [ ]:
# Demo: how d_k affects the distribution of softmax outputs
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(42)
for d_k in [4, 64, 512]:
    q = rng.standard_normal(d_k)
    k = rng.standard_normal((8, d_k))
    raw   = softmax(q @ k.T)
    scaled = softmax(q @ k.T / np.sqrt(d_k))
    print(f'd_k={d_k:4d} | unscaled max weight={raw.max():.3f}  scaled max weight={scaled.max():.3f}')

## 3. Multi-head attention

A single attention head can only learn one kind of "relatedness". What multi-head does:

```
split d_model into h parts, each of dimension d_k = d_model / h
head_i = Attention(Q·Wq_i, K·Wk_i, V·Wv_i)   # independent linear projections
output = Concat(head_0, ..., head_{h-1}) · Wo  # concat, then project
```

Each head does attention in a different subspace, so the heads can attend to different patterns at the same time — "grammatical dependencies", "semantic similarity", "positional nearness" and so on — and a final linear transform fuses them.

This lesson implements only the single-head core `scaled_dot_product_attention`; the multi-head projection matrices `Wq/Wk/Wv/Wo` will be wrapped into a `MultiHeadAttention` class in the planned Whisper encoder (`aurora/whisper/attention.py`, a planned module not created yet). The next lesson, L84, turns to LoRA fine-tuning and doesn't unpack the multi-head wrapper any further.

In [ ]:
# Demo: the multi-head idea — splitting d_model=16 into h=2 heads
d_model, h = 16, 2
d_k = d_model // h
rng = np.random.default_rng(7)
x = rng.standard_normal((1, 5, d_model))  # (B=1, seq=5, d_model=16)

# Simplification: simulate the projections with slices (a real implementation uses Wq @ x)
for i in range(h):
    Q_h = x[:, :, i*d_k:(i+1)*d_k]
    K_h = x[:, :, i*d_k:(i+1)*d_k]
    V_h = x[:, :, i*d_k:(i+1)*d_k]
    print(f'head {i}: Q shape {Q_h.shape}, d_k={d_k}')

## 4. ✏️ Implement `scaled_dot_product_attention(Q, K, V, mask=None)`

**Signature**:
```python
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q: (B, seq_q, d_k)
    # K: (B, seq_k, d_k)
    # V: (B, seq_k, d_v)
    # mask: (B, seq_q, seq_k) bool; scores at True positions are set to -inf (optional)
    # returns: (B, seq_q, d_v) float
```

**A 4-step route**:

| Step | Operation | Formula/code |
|---|---|---|
| 1 | Compute the scores | `Q @ K.transpose(0,2,1) / sqrt(d_k)` → (B, seq_q, seq_k) |
| 2 | Apply the mask (optional) | `scores[mask] = -1e9` (or -inf) |
| 3 | Softmax normalization | along `axis=-1` (the seq_k axis) |
| 4 | Weighted aggregation of V | `weights @ V` → (B, seq_q, d_v) |

**Acceptance criteria**:
- Output shape = `(B, seq_q, d_v)`
- Each row of `weights` sums to 1.0 (softmax normalization)
- `Q==K` (with all vectors the same length): each position's attention weight is largest on itself (self-attention points to itself)

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, seq_q, d_k)
    K: (B, seq_k, d_k)
    V: (B, seq_k, d_v)
    mask: bool array (B, seq_q, seq_k), True means that position is masked out
    returns: (B, seq_q, d_v)
    """
    d_k = Q.shape[-1]
    # ✏️ TODO: compute the scaled dot-product scores, shape (B, seq_q, seq_k)
    raise NotImplementedError("TODO: scores = Q @ K.T / np.sqrt(d_k)")

    # ✏️ TODO: if mask is not None, mask out those positions (set to -inf)

    # ✏️ TODO: softmax(scores, axis=-1) → weights
    raise NotImplementedError("TODO: weights = softmax(scores)")

    # ✏️ TODO: return weights @ V
    raise NotImplementedError("TODO: return weights @ V")

In [ ]:
# Shape check (silently skipped while scaled_dot_product_attention is not implemented)
np.random.seed(1)
Q = np.random.randn(2, 4, 8)
K = np.random.randn(2, 6, 8)
V = np.random.randn(2, 6, 16)

try:
    out = scaled_dot_product_attention(Q, K, V)
    if out is None:
        print('⬜ Please implement the 4 TODO steps of scaled_dot_product_attention first')
    else:
        assert out.shape == (2, 4, 16), f'wrong shape: {out.shape}'
        print('✅ output shape:', out.shape)

        # Check that the weight rows sum to 1
        d_k = Q.shape[-1]
        scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
        scores_s = scores - scores.max(axis=-1, keepdims=True)
        w = np.exp(scores_s); w /= w.sum(axis=-1, keepdims=True)
        assert np.allclose(w.sum(axis=-1), 1.0, atol=1e-6)
        print('✅ every row of the softmax weights sums to 1')

        # Causal mask check
        B, seq = 1, 4
        Q2 = np.random.randn(B, seq, 8)
        K2 = np.random.randn(B, seq, 8)
        V2 = np.random.randn(B, seq, 8)
        mask = np.triu(np.ones((B, seq, seq), dtype=bool), k=1)
        out_masked = scaled_dot_product_attention(Q2, K2, V2, mask=mask)
        assert out_masked.shape == (B, seq, 8)
        print('✅ causal-mask output shape is correct:', out_masked.shape)

        # Causal-mask semantics check: the student's output must match the reference implementation
        scores_m = Q2 @ K2.transpose(0, 2, 1) / np.sqrt(d_k)
        scores_m[mask] = -1e9          # mask out future positions (a large negative number instead of -inf)
        e_m = np.exp(scores_m - scores_m.max(axis=-1, keepdims=True))
        w_m = e_m / e_m.sum(axis=-1, keepdims=True)
        ref_masked = w_m @ V2
        # pos=0 can only attend to pos=0; all other weights should be ≈ 0
        assert w_m[0, 0, 1:].max() < 1e-6, \
            f'causal-mask reference weights are not as expected: {w_m[0, 0, 1:]}'
        assert np.allclose(out_masked, ref_masked, atol=1e-4), \
            'causal mask not applied correctly: the student output does not match the reference implementation'
        print('✅ causal-mask semantics check passed (future-position weights ≈ 0, output matches the reference)')
except (NotImplementedError, TypeError):
    print('⬜ Please finish the 4 TODO steps of scaled_dot_product_attention first')
except TypeError:
    print('⬜ The function returned something that is not an array; please finish every TODO')

## 5. Parameter experiment: how the direction of Q affects the attention weights

Fix `K` (3 keys) and `V`, change the direction of `Q`, and watch how `weights` changes with the angle between `Q` and each `K`.

**What you should see**:
- Set `Q` exactly parallel to `K[0]` (`Q = K[0]`) → `weights[0]` close to 1
- Set `Q` to a "neutral" direction (`Q` at the same angle to every `K`) → `weights` tends to the uniform distribution `[1/3, 1/3, 1/3]`
- The larger `d_k`, the more extreme (sharper) the weights get without scaling

In [ ]:
np.random.seed(3)
d_k = 8
K_exp = np.random.randn(1, 3, d_k)  # 3 keys
V_exp = np.random.randn(1, 3, d_k)

# Experiment 1: Q parallel to K[0]
Q_parallel = K_exp[:, [0], :]  # shape (1,1,d_k)
try:
    out1 = scaled_dot_product_attention(Q_parallel, K_exp, V_exp)
except (NotImplementedError, TypeError):
    out1 = None  # if not implemented yet, this doesn't affect the hand-computed weight demo below
scores1 = Q_parallel @ K_exp.transpose(0,2,1) / np.sqrt(d_k)
s1 = scores1 - scores1.max(axis=-1, keepdims=True)
w1 = np.exp(s1); w1 /= w1.sum(axis=-1, keepdims=True)
print('Q ∥ K[0] → weights:', w1[0,0].round(3))

# Experiment 2: Q is the zero vector (uniform attention)
Q_zero = np.zeros((1, 1, d_k))
scores2 = Q_zero @ K_exp.transpose(0,2,1) / np.sqrt(d_k)
s2 = scores2 - scores2.max(axis=-1, keepdims=True)
w2 = np.exp(s2); w2 /= w2.sum(axis=-1, keepdims=True)
print('Q = 0 (uniform) → weights:', w2[0,0].round(3))

# Experiment 3: how d_k affects sharpness
print('\n--- How d_k affects sharpness (unscaled vs scaled) ---')
for dk in [4, 64, 512]:
    rng2 = np.random.default_rng(99)
    q_ = rng2.standard_normal((1,1,dk))
    k_ = rng2.standard_normal((1,3,dk))
    sc_raw = q_ @ k_.transpose(0,2,1)
    sc_scl = sc_raw / np.sqrt(dk)
    def sw(s):
        s = s - s.max(axis=-1, keepdims=True)
        e = np.exp(s); return e / e.sum(axis=-1, keepdims=True)
    print(f'  d_k={dk:4d}: unscaled max={sw(sc_raw)[0,0].max():.3f}  scaled max={sw(sc_scl)[0,0].max():.3f}')

## 6. Positional encoding (PE)

The attention mechanism is naturally blind to position in the sequence (it only looks at content similarity). PE adds position information into the word embeddings:

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

- Each position `pos` maps to a unique d_model-dimensional vector
- The dot product between PEs of nearby positions decays smoothly with distance, which helps the model learn relative position
- Sine and cosine alternate: even dimensions use sin, odd dimensions use cos

Once PE is added to the input x, attention can tell apart the same word appearing at different positions.

In [ ]:
import numpy as np

def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, None]            # (seq_len, 1)
    i   = np.arange(0, d_model, 2)[None, :]     # (1, d_model//2)
    denom = 10000 ** (i / d_model)
    pe[:, 0::2] = np.sin(pos / denom)           # even dims: sin
    pe[:, 1::2] = np.cos(pos / denom)           # odd dims: cos
    return pe

# Verify
seq_len, d_model = 8, 16
pe = positional_encoding(seq_len, d_model)
assert pe.shape == (seq_len, d_model)
# Even columns should be sin: |sin|<=1
assert np.all(np.abs(pe[:, 0::2]) <= 1.0 + 1e-9)
# Odd columns should be cos: |cos|<=1
assert np.all(np.abs(pe[:, 1::2]) <= 1.0 + 1e-9)
print(f'positional_encoding({seq_len}, {d_model}) shape:', pe.shape)
print(f'Row 0 (pos=0): {np.round(pe[0, :8], 3)}')
print(f'Row 1 (pos=1): {np.round(pe[1, :8], 3)}')
print('✅ PE shape and value-range checks passed')


## 7. The feed-forward network (FFN)

Every Transformer block follows its attention layer with a two-layer MLP:

```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
```

- The inner dimension (`d_ff`) is usually 4 times `d_model`: expand, then compress
- It is applied to each position of the sequence **independently** (position-wise)
- Working with the attention layer: attention handles communication across positions, the FFN handles the feature transform at each position

Whisper's d_model=512, d_ff=2048 (4×); GPT-2's d_model=768, d_ff=3072 (4×).

In [ ]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

class FFN:
    def __init__(self, d_model, d_ff, rng=None):
        if rng is None:
            rng = np.random.default_rng(0)
        scale = np.sqrt(2.0 / d_model)
        self.W1 = rng.normal(0, scale, (d_model, d_ff))
        self.b1 = np.zeros(d_ff)
        self.W2 = rng.normal(0, scale, (d_ff, d_model))
        self.b2 = np.zeros(d_model)

    def __call__(self, x):
        # x: (seq_len, d_model)  or  (d_model,)
        h = relu(x @ self.W1 + self.b1)
        return h @ self.W2 + self.b2

# Verify
d_model, d_ff, seq_len = 16, 64, 5
ffn = FFN(d_model, d_ff)
x_in = np.random.randn(seq_len, d_model)
x_out = ffn(x_in)
assert x_out.shape == (seq_len, d_model), f'wrong output shape: {x_out.shape}'
print(f'FFN({d_model}, {d_ff}): ({seq_len},{d_model}) -> {x_out.shape}')
print('✅ FFN shape check passed (a position-wise transform; input and output dimensions are the same)')


## 🎯 This foundation will keep coming back

The little 3×3 attention you just worked by hand will show up again and again in **L84 / L85 / L90 / L91** — L84's LoRA fine-tunes it, L85's KV-Cache speeds it up, L90's agent reasons with it, and L91 draws its weights as a heatmap. The whole age of large models grows out of this one `softmax(QKᵀ/√d_k)·V`, so it's worth welding it into muscle memory right now.

## 8. Putting it together: attention + PE + FFN = the core of a Transformer block

```
x_emb = embedding(tokens) + positional_encoding(seq_len, d_model)
x_att = scaled_dot_product_attention(Q=x_emb, K=x_emb, V=x_emb)
x_out = ffn(x_att + x_emb)  # residual connection
```

A real Transformer block also includes LayerNorm, which we leave out here.
The next lesson (L84) builds on this to implement a LoRA low-rank adaptation layer.

In [ ]:
import numpy as np

# ── Reference implementation (only for comparison checks; it does not replace the student's function) ────────────────────────
def softmax_2d(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _ref_sdpa(Q, K, V):
    """Reference implementation for comparing with the student's; it does not replace scaled_dot_product_attention"""
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
    return softmax_2d(scores) @ V

# Mini transformer block parameters
rng = np.random.default_rng(42)
d_model, d_ff, seq = 16, 64, 5

x = rng.standard_normal((seq, d_model))
pe = positional_encoding(seq, d_model)
x_pe = x + pe                          # add positional encoding

# ── Combined check: calls the student's scaled_dot_product_attention ───────────
try:
    x_att = scaled_dot_product_attention(x_pe[None], x_pe[None], x_pe[None])[0]
    x_res = x_att + x_pe               # residual connection
    ffn = FFN(d_model, d_ff, rng)
    x_out = ffn(x_res) + x_res        # FFN + residual

    assert x_out.shape == (seq, d_model), f'wrong shape: {x_out.shape}'

    # Compare with the reference implementation (same random seed)
    ref_rng = np.random.default_rng(42)
    x_ref = ref_rng.standard_normal((seq, d_model))
    x_pe_ref = x_ref + pe
    x_att_ref = _ref_sdpa(x_pe_ref[None], x_pe_ref[None], x_pe_ref[None])[0]
    x_res_ref = x_att_ref + x_pe_ref
    ffn_ref = FFN(d_model, d_ff, ref_rng)
    x_out_ref = ffn_ref(x_res_ref) + x_res_ref
    assert np.allclose(x_out, x_out_ref, atol=1e-6), \
        'the student implementation does not match the reference; check scaled_dot_product_attention'

    print(f'Transformer Block: ({seq},{d_model}) -> ({seq},{d_model})')
    print('✅ Full attention + PE + FFN forward pass passed (using the student implementation)')
except (NotImplementedError, TypeError):
    print('⬜ Please finish the 4 TODO steps of scaled_dot_product_attention before running this check')
except (TypeError, AttributeError) as e:
    print(f'⬜ The function returned the wrong type; please finish every TODO: {e}')


## Wrap-up

This lesson first took apart attention's three roles, the scaling and the division of labor between heads, then tied them together with one minimal example. What you take away isn't a lonely formula but an intuition: "compare similarity first, then fetch content by weight". Next lesson (**L84**) sticks a small low-rank patch beside these same linear layers and carries on toward LoRA fine-tuning.

## ✏️ Closed-book derivation check cell — tracking dimensions through multi-head attention

**Rules: close every cell above, fill in the table in Markdown first, then run the code cell to check. If the output doesn't match what you expected, fix your answer in this Markdown cell and run again — don't look at the code cell's output and then fill in the table.**

**The problem**: multi-head attention with batch=1, seq_len=T=20, d_model=512, h=8 heads, d_k=d_v=64

Track the tensor shapes step by step (fill in the blanks):

| Step | Operation | Output shape |
|------|------|---------|
| 1 | Input X | (1, 20, 512) |
| 2 | Q = X @ W_Q | (1, 20, ___) |
| 3 | reshape → multi-head Q | (1, ___, 20, ___) |
| 4 | scores = Q @ K^T / √d_k | (1, ___, 20, ___) |
| 5 | weights = softmax(scores) | (1, ___, 20, ___) |
| 6 | attn = weights @ V | (1, ___, 20, ___) |
| 7 | concat → (1, 20, ___) |  |
| 8 | output = concat @ W_O | (1, 20, ___) |

Write out the matrix multiplication of step 4 (scores = Q @ K^T): what is the shape of K^T? Why divide by √d_k?

(Fill in the table and write your explanation here...)

In [ ]:
# Check: run this and see whether your answers are right
# If a row doesn't match, go back to the Markdown cell, fix your answer and run again (don't look at the output first)
import numpy as np

B, T, D_MODEL, H = 1, 20, 512, 8
D_K = D_MODEL // H   # 64

np.random.seed(0)
X  = np.random.randn(B, T, D_MODEL)
Wq = np.random.randn(D_MODEL, D_MODEL) * 0.02
Wk = np.random.randn(D_MODEL, D_MODEL) * 0.02
Wv = np.random.randn(D_MODEL, D_MODEL) * 0.02
Wo = np.random.randn(D_MODEL, D_MODEL) * 0.02

Q = (X @ Wq).reshape(B, T, H, D_K).transpose(0,2,1,3)
K = (X @ Wk).reshape(B, T, H, D_K).transpose(0,2,1,3)
V = (X @ Wv).reshape(B, T, H, D_K).transpose(0,2,1,3)
scores  = Q @ K.transpose(0,1,3,2) / np.sqrt(D_K)
weights = np.exp(scores - scores.max(-1, keepdims=True))
weights /= weights.sum(-1, keepdims=True)
attn    = weights @ V
concat  = attn.transpose(0,2,1,3).reshape(B, T, D_MODEL)
out     = concat @ Wo

expected = {
    "Q (after W_Q)":      (B, T, D_MODEL),
    "Q (multi-head)":     (B, H, T, D_K),
    "scores (Q @ K^T)":   (B, H, T, T),
    "weights (softmax)":  (B, H, T, T),
    "attn (w @ V)":       (B, H, T, D_K),
    "concat":             (B, T, D_MODEL),
    "output":             (B, T, D_MODEL),
}
actual = {
    "Q (after W_Q)":      (X @ Wq).shape,
    "Q (multi-head)":     Q.shape,
    "scores (Q @ K^T)":   scores.shape,
    "weights (softmax)":  weights.shape,
    "attn (w @ V)":       attn.shape,
    "concat":             concat.shape,
    "output":             out.shape,
}

all_ok = True
for name, exp in expected.items():
    act = actual[name]
    ok  = act == exp
    all_ok = all_ok and ok
    print(f"  {'✅' if ok else '❌'} {name}: {act}  (expected {exp})")

if all_ok:
    print("\n✅ Every dimension is correct!")
else:
    print("\n❌ Some dimensions don't match — go back to the Markdown cell and fix your table; don't just copy the output")

In [ ]:
# ✏️ Self-review for this lesson
l83_review = {
    "qkv_intuition":              None,  # understand the roles of Q/K/V: Q asks, K answers, V gets read? True/False
    "sqrt_dk_rationale":          None,  # understand that dividing by √d_k stops the variance blowing up → softmax saturating? True/False
    "attention_impl":             None,  # scaled_dot_product_attention() is correct; shape and weight normalization pass? True/False
    "multihead_concept":          None,  # understand multi-head: h heads run independently and capture different dependencies? True/False
    "whiteboard_passed":          None,  # cleared the whiteboard challenge tracking multi-head attention dimensions? True/False
}

unfilled = [k for k, v in l83_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l83_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L83 all clear! On to L84: LoRA')

---

→ **Next lesson** · [L84 · LoRA](L84_lora.ipynb)

> Next lesson covers **LoRA**: W = W₀ + BA, fine-tuning a GPT-style model with 0.1% of the parameters.